In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    "./models/roberta_base"
)

device = torch.device("cuda")
model = model.to(device)

pretrained_model = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [2]:
# Inference: test custom sentences
texts = [
    "This movie was surprisingly good and uplifting.",
    "I hated the ending; it was a complete waste of time.",
    "The plot was okay, but the acting felt flat.",
]

# Update these to your label names in index order
label_names = ["negative", "neutral", "positive"]

# Use the fine-tuned model already in memory
model.eval()

enc = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
)
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    logits = model(**enc).logits
    preds = torch.argmax(logits, dim=-1).cpu().tolist()

decoded = [label_names[p] if p < len(label_names) else f"label_{p}" for p in preds]
print(list(zip(texts, decoded)))

[('This movie was surprisingly good and uplifting.', 'positive'), ('I hated the ending; it was a complete waste of time.', 'negative'), ('The plot was okay, but the acting felt flat.', 'neutral')]
